# scGPT-spatial Benchmarking

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 10.02.2025
- Date of Last Modification: 24.12.2025 (Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the scgpt-spatial conda environment, installable from ```../../envs/scgpt-spatial.yaml```
- Clone scGPT-spatial from GitHub into ```<root>/analysis/benchmarking```: ```git clone https://github.com/bowang-lab/scGPT-spatial```
- Make a small code modification in ```<root>/analysis/benchmarking/scGPT-spatial/scgpt_spatial/tasks/cell_emb``` line 132: change ```np.float``` to ```float```.
- Load conda 11.7.1: ```source /etc/profile.d/modules.sh``` -> ```module load cuda-11.7.1``` (on gpu-lotfollahi queue)
- Install torch and flash-attn:
    - pip install torch==2.0.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
    - pip install torchtext==0.15.2
    - pip install flash-attn==1.0.4
- Download the checkpoint (scGPT_spatial_v1) from https://figshare.com/articles/software/scGPT-spatial_V1_Model_Weights/28356068?file=52163879 as described at https://github.com/bowang-lab/scGPT-spatial/blob/main/tutorials/Multi_Modal_Integration_Demo.ipynb and store in ```<root>/analysis/benchmarking/scGPT-spatial/checkpoints```.
- This notebook follows the tutorial from https://github.com/bowang-lab/scGPT-spatial/blob/main/tutorials/Multi_Modal_Integration_Demo.ipynb.

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")
sys.path.insert(0, "./scGPT-spatial")

In [ ]:
import os
import sys
import warnings
from pathlib import Path
from tqdm import tqdm
from typing import Dict, Optional, Union

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scib
import torch
from matplotlib import pyplot as plt
from scipy.sparse import csr_matrix
from scipy.stats import mode
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

import scgpt_spatial
from scgpt_spatial.utils import eval_scib_metrics

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'
model_dir = './scGPT-spatial/checkpoints/scGPT_spatial_v1'

### 1.3 Define Config

In [ ]:
min_cells_per_niche = 100
gene_col = 'feature_name'
latent_neighborhood_cluster_key = 'predicted_niche'
latent_cell_cluster_key = 'predicted_cell_type'

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/scGPT-spatial', exist_ok=True)

## 2. Zero-shot Niche Identification

### 2.1 chl59-8b_1p (NanoString CosMx Human NSCLC)

#### 2.1.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'chl59-8b_1p'
batches = ['Lung9_Rep1',
           'Lung9_Rep2',
           'Lung5_Rep1',
           'Lung5_Rep2',
           'Lung6',
           'Lung12']
niche_key = 'niche'
batch_key = 'batch'
spot_sizes = [80 * 0.18] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.18 # conversion rate from https://nanostring-public-share.s3.us-west-2.amazonaws.com/SMI-Compressed/SMI-ReadMe.html

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 2.1.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obs[batch_key] = adata.obs[batch_key]
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=niche_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 2.1.3 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells - 12)
latent_leiden_resolution = 0.5

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_neighborhood_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_neighborhood_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_neighborhood_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_neighborhood_cluster_key].unique().tolist())
print(filtered_df[latent_neighborhood_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_neighborhood_cluster_key)

In [ ]:
inferred_niches_ordered = [
    '0',
    '2',
    '7',
    '6',
    '11',
    '8',
    '5',
    '1',
    '3',
    '4',
    '9',
    '10']

In [ ]:
# Plot scGPT-spatial niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_neighborhood_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_neighborhood_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.1.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_neighborhood_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_neighborhood_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.2 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.2.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'chr78-11b_1p'
batches = ['CM007',
           'CM008',
           'CM009',
           'CM011',
           'CM013',
           'CM015',
           'CM016',
           'CM017',
           'CM018',
           'CM019',
           'CM020']
niche_key = 'broad_anatomy'
batch_key = 'batch'
spot_sizes = [240 * 0.093] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.093 # conversion rate from Valentina

In [ ]:
# Combine regions
adata.obs[niche_key] = adata.obs[niche_key].apply(lambda x: x.replace('Urogenital Sinus', 'UrogenitalSinus'))

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 2.2.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obs[batch_key] = adata.obs[batch_key]
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=niche_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 2.2.3 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 19)
latent_leiden_resolution = 0.9

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_neighborhood_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_neighborhood_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_neighborhood_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_neighborhood_cluster_key].unique().tolist())
print(filtered_df[latent_neighborhood_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_neighborhood_cluster_key)

In [ ]:
inferred_niches_ordered = [
    '12',
    '1',
    '4',
    '9',
    '5',
    '10',
    '8',
    '17',
    '13',
    '3',
    '16',
    '15',
    '18',
    '2',
    '0',
    '11',
    '7',
    '6',
    '14',]

In [ ]:
# Plot scGPT-spatial niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_neighborhood_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_neighborhood_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.2.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_neighborhood_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_neighborhood_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.3 xhs49-1b_1p (Xenium Human Melanoma)

#### 2.3.1 Load Data

In [ ]:
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
spot_size = 20

In [ ]:
adata = sc.read_h5ad(f'{st_data_silver_folder_path}/{dataset_name}/Xenium_Prime_Human_Skin_FFPE_outs.h5ad')

# Filter cells without niche labels
adata = adata[~adata.obs['niche_name'].isna()]

In [ ]:
adata.obs['niche_name'].value_counts()

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 2.3.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=niche_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 2.3.3 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells - 7)
latent_leiden_resolution = 0.15

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_neighborhood_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_neighborhood_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_neighborhood_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_neighborhood_cluster_key].unique().tolist())
print(filtered_df[latent_neighborhood_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_neighborhood_cluster_key)

In [ ]:
inferred_niches_ordered = [
    '6',
    '4',
    '1',
    '2',
    '5',
    '0',
    '3',]

In [ ]:
# Plot scGPT-spatial niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

file_name = f'{dataset_name}_scgpt-spatial_niches.svg'
sc.pl.spatial(adata,
              color=latent_neighborhood_cluster_key,
              palette=pred_palette,
              spot_size=spot_size,
              save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_neighborhood_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.3.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_neighborhood_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_neighborhood_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 2.4 xhs1000-39b_1p (Xenium Human Skin - Eczema)

#### 2.4.1 Load Data

In [ ]:
# Define dataset params
dataset_name = 'xhs1000-39b_1p'
batches = ['batch11',
           'batch19',
           'batch32']
niche_key = 'niche_type'
batch_key = 'sample'
spot_sizes = [20] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial']

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 2.4.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obs[batch_key] = adata.obs[batch_key]
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=niche_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 2.4.3 Highlight Run (Matching Ground Truth Number of Niches)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells - 11)
latent_leiden_resolution = 0.3

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_neighborhood_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
niche_counts = (adata.obs[niche_key].value_counts() < 100)
invalid_niches = niche_counts[niche_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_neighborhood_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[niche_key].isin(invalid_niches)) & (~adata.obs[latent_neighborhood_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_neighborhood_cluster_key].unique().tolist())
print(filtered_df[latent_neighborhood_cluster_key].value_counts())

max_overlap_plot(filtered_df, niche_key, latent_neighborhood_cluster_key)

In [ ]:
inferred_niches_ordered = [
    '1',
    '3',
    '0',
    '9',
    '5',
    '4',
    '2',
    '7',
    '8',
    '10',
    '6',]

In [ ]:
# Plot scGPT-spatial niches
label2niche = {}
current_index = 0  # To keep track of the new indices

for niche in inferred_niches_ordered:
    if not isinstance(niche, str):
        if math.isnan(niche):
            continue # skip NaN values
    label2niche[str(current_index)] = niche
    current_index += 1

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_neighborhood_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[niche_key]
labels_pred = adata.obs[latent_neighborhood_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 2.4.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_niches_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_neighborhood_cluster_key,
                 resolution=latent_leiden_resolution)

    n_niches_list.append(adata.obs[latent_neighborhood_cluster_key].nunique())
    
    labels = adata.obs[niche_key]
    labels_pred = adata.obs[latent_neighborhood_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_niches': n_niches_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

## 3. Zero-shot Cell Type Identification

### 3.1 xhs1000-39b_1p (Xenium Human Eczema)

In [ ]:
# Define dataset params
dataset_name = 'xhs1000-39b_1p'
batches = ['batch11',
           'batch19',
           'batch32']
cell_type_key = 'new_annotation' # 'annotation_new7'
batch_key = 'batch'
spot_sizes = [20] * len(batches)

#### 3.1.1 Load Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial']

In [ ]:
print(adata.obs[cell_type_key].value_counts())

#### 3.1.2 Load Embeddings

In [ ]:
# Load embeddings from 2.4
adata = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 3.1.3 Highlight Run (Matching Ground Truth Number of Cell Types)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 19)
latent_leiden_resolution = 0.75

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_cell_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
print(adata.obs[latent_cell_cluster_key].value_counts())

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered_cell_type.h5ad')

In [ ]:
cell_type_counts = (adata.obs[cell_type_key].value_counts() < 100)
invalid_cell_types = cell_type_counts[cell_type_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cell_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[cell_type_key].isin(invalid_cell_types)) & (~adata.obs[latent_cell_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cell_cluster_key].unique().tolist())
print(filtered_df[latent_cell_cluster_key].value_counts())

max_overlap_plot(filtered_df, cell_type_key, latent_cell_cluster_key)

In [ ]:
inferred_cell_types_ordered = [
    '16',
    '6',
    '2',
    '12',
    '4',
    '8',
    '10',
    '11',
    '7',
    '5',
    '14',
    '18',
    '13',
    '17',
    '3',
    '9',
    '1',
    '0',
    '15',]

In [ ]:
# Plot scGPT-spatial cell types
label2celltype = {}
current_index = 0  # To keep track of the new indices

for cell_type in inferred_cell_types_ordered:
    if not isinstance(cell_type, str):
        if math.isnan(cell_type):
            continue # skip NaN values
    label2celltype[str(current_index)] = cell_type
    current_index += 1

pred_palette = {
    label2celltype.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_cell_types.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cell_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[cell_type_key]
labels_pred = adata.obs[latent_cell_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 3.1.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_cell_cluster_key,
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())
    
    labels = adata.obs[cell_type_key]
    labels_pred = adata.obs[latent_cell_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_cell_types': n_cell_types_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores_cell_type.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 3.2 chl59-8b_1p (NanoString CosMx Human NSCLC)

In [ ]:
# Define dataset params
dataset_name = 'chl59-8b_1p'
batches = ['Lung9_Rep1',
           'Lung9_Rep2',
           'Lung5_Rep1',
           'Lung5_Rep2',
           'Lung6',
           'Lung12']
cell_type_key = 'cell_type'
batch_key = 'batch'
spot_sizes = [80 * 0.18] * len(batches)

#### 3.2.1 Load Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without cell type labels
            adata = adata[~adata.obs[cell_type_key].isna()]

            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.18 # conversion rate from https://nanostring-public-share.s3.us-west-2.amazonaws.com/SMI-Compressed/SMI-ReadMe.html

In [ ]:
print(adata.obs[cell_type_key].value_counts())

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 3.2.2 Load Embeddings

In [ ]:
# Load embeddings from 2.1
adata = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 3.2.3 Highlight Run (Matching Ground Truth Number of Cell Types)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 10)
latent_leiden_resolution = 0.4

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_cell_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
print(adata.obs[latent_cell_cluster_key].value_counts())

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered_cell_type.h5ad')

In [ ]:
cell_type_counts = (adata.obs[cell_type_key].value_counts() < 100)
invalid_cell_types = cell_type_counts[cell_type_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cell_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[cell_type_key].isin(invalid_cell_types)) & (~adata.obs[latent_cell_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cell_cluster_key].unique().tolist())
print(filtered_df[latent_cell_cluster_key].value_counts())

max_overlap_plot(filtered_df, cell_type_key, latent_cell_cluster_key)

In [ ]:
inferred_cell_types_ordered = [
    '4',
    '2',
    '6',
    '8',
    '3',
    '9',
    '5',
    '1',
    '7',
    '0',]

In [ ]:
# Plot scGPT-spatial cell types
label2celltype = {}
current_index = 0  # To keep track of the new indices

for cell_type in inferred_cell_types_ordered:
    if not isinstance(cell_type, str):
        if math.isnan(cell_type):
            continue # skip NaN values
    label2celltype[str(current_index)] = cell_type
    current_index += 1

pred_palette = {
    label2celltype.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_cell_types.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cell_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[cell_type_key]
labels_pred = adata.obs[latent_cell_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 3.2.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_cell_cluster_key,
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())
    
    labels = adata.obs[cell_type_key]
    labels_pred = adata.obs[latent_cell_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_cell_types': n_cell_types_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores_cell_type.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 3.3 xhk1007-CV1-41b_1p (Xenium Human Kidney)

In [ ]:
# Define dataset params
dataset_name = 'xhk1007-CV1-41b_1p'
batches = ['CV1-KID-0-FT-1', # batch14
           'CV1-KID-0-FT-3', # batch1
          ]
cell_type_key = 'level_3_cell_type'
batch_key = 'batch'
spot_sizes = [30] * len(batches)

#### 3.3.1 Load Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        print(file)
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without cell type labels
            adata = adata[~adata.obs[cell_type_key].isna()]

            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata.obs[cell_type_key].value_counts()

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 3.3.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obs[batch_key] = adata.obs[batch_key]
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=cell_type_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 3.3.3 Highlight Run (Matching Ground Truth Number of Cell Types)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 13)
latent_leiden_resolution = 0.45

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_cell_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
print(adata.obs[latent_cell_cluster_key].value_counts())

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered_cell_type.h5ad')

In [ ]:
cell_type_counts = (adata.obs[cell_type_key].value_counts() < 100)
invalid_cell_types = cell_type_counts[cell_type_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cell_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[cell_type_key].isin(invalid_cell_types)) & (~adata.obs[latent_cell_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cell_cluster_key].unique().tolist())
print(filtered_df[latent_cell_cluster_key].value_counts())

max_overlap_plot(filtered_df, cell_type_key, latent_cell_cluster_key)

In [ ]:
inferred_cell_types_ordered = [
    '7',
    '12',
    '1',
    '5',
    '4',
    '11',
    '9',
    '10',
    '6',
    '3',
    '2',
    '0',
    '8',]

In [ ]:
# Plot scGPT-spatial cell types
label2celltype = {}
current_index = 0  # To keep track of the new indices

for cell_type in inferred_cell_types_ordered:
    if not isinstance(cell_type, str):
        if math.isnan(cell_type):
            continue # skip NaN values
    label2celltype[str(current_index)] = cell_type
    current_index += 1

pred_palette = {
    label2celltype.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_cell_types.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cell_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[cell_type_key]
labels_pred = adata.obs[latent_cell_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 3.3.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_cell_cluster_key,
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())
    
    labels = adata.obs[cell_type_key]
    labels_pred = adata.obs[latent_cell_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_cell_types': n_cell_types_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores_cell_type.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 3.4 shp75-4b_1p (STARmap Human Placenta)

In [ ]:
# Define dataset params
dataset_name = 'shp75-4b_1p'
batches = ['STARmap-ISS_sample_W11_',
           'STARmap-ISS_sample_W7_',
           'STARmap-ISS_sample_W8-2_',
           'STARmap-ISS_sample_W9_',
          ]
cell_type_key = 'leiden'
batch_key = 'batch'
spot_sizes = [30] * len(batches)

#### 3.4.1 Load Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        print(file)
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without cell type labels
            adata = adata[~adata.obs[cell_type_key].isna()]

            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata.obs[cell_type_key].value_counts()

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 3.4.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obs[batch_key] = adata.obs[batch_key]
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=cell_type_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 3.4.3 Highlight Run (Matching Ground Truth Number of Cell Types)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 13)
latent_leiden_resolution = 1.0

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_cell_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
print(adata.obs[latent_cell_cluster_key].value_counts())

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered_cell_type.h5ad')

In [ ]:
cell_type_counts = (adata.obs[cell_type_key].value_counts() < 100)
invalid_cell_types = cell_type_counts[cell_type_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cell_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[cell_type_key].isin(invalid_cell_types)) & (~adata.obs[latent_cell_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cell_cluster_key].unique().tolist())
print(filtered_df[latent_cell_cluster_key].value_counts())

max_overlap_plot(filtered_df, cell_type_key, latent_cell_cluster_key)

In [ ]:
inferred_cell_types_ordered = [
    '2',
    '8',
    '1',
    '4',
    '0',
    '5',
    '3',
    '7',
    '6',
    '9',
    '11',
    '10',
    '12',]

In [ ]:
# Plot scGPT-spatial cell types
label2celltype = {}
current_index = 0  # To keep track of the new indices

for cell_type in inferred_cell_types_ordered:
    if not isinstance(cell_type, str):
        if math.isnan(cell_type):
            continue # skip NaN values
    label2celltype[str(current_index)] = cell_type
    current_index += 1

pred_palette = {
    label2celltype.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_cell_types.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cell_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[cell_type_key]
labels_pred = adata.obs[latent_cell_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 3.4.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_cell_cluster_key,
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())
    
    labels = adata.obs[cell_type_key]
    labels_pred = adata.obs[latent_cell_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_cell_types': n_cell_types_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores_cell_type.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 3.5 xhp1016-18b_1p (Xenium Human Pancreas)

In [ ]:
# Define dataset params
dataset_name = 'xhp1016-18b_1p'
batches = ['output-XETG00055__0073068__Hml4-PNC-0-FFPE-1-s4__20250814__132610', # batch3
           'output-XETG00055__0073075__Hml7-PNC-0-FFPE-1-s4__20250814__132610', # batch12
          ]
cell_type_key = 'Annotation_high'
batch_key = 'batch'
spot_sizes = [30] * len(batches)

#### 3.5.1 Load Data

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        print(file)
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without cell type labels
            adata = adata[~adata.obs[cell_type_key].isna()]

            adata.obs[batch_key] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata.obs[cell_type_key].value_counts()

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 3.5.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obs[batch_key] = adata.obs[batch_key]
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=cell_type_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 3.5.3 Highlight Run (Matching Ground Truth Number of Cell Types)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 16)
latent_leiden_resolution = 0.6

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_cell_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
print(adata.obs[latent_cell_cluster_key].value_counts())

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered_cell_type.h5ad')

In [ ]:
cell_type_counts = (adata.obs[cell_type_key].value_counts() < 100)
invalid_cell_types = cell_type_counts[cell_type_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cell_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[cell_type_key].isin(invalid_cell_types)) & (~adata.obs[latent_cell_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cell_cluster_key].unique().tolist())
print(filtered_df[latent_cell_cluster_key].value_counts())

max_overlap_plot(filtered_df, cell_type_key, latent_cell_cluster_key)

In [ ]:
inferred_cell_types_ordered = [
    '7',
    '12',
    '6',
    '15',
    '3',
    '4',
    '13',
    '1',
    '11',
    '10',
    '8',
    '9',
    '0',
    '14',
    '5',
    '2',]

In [ ]:
# Plot scGPT-spatial cell types
label2cell_type = {}
current_index = 0  # To keep track of the new indices

for cell_type in inferred_cell_types_ordered:
    if not isinstance(cell_type, str):
        if math.isnan(cell_type):
            continue # skip NaN values
    label2cell_type[str(current_index)] = cell_type
    current_index += 1

pred_palette = {
    label2cell_type.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_scgpt-spatial_cell_types.svg'
    sc.pl.spatial(adata_batch,
                  color=latent_cell_cluster_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[cell_type_key]
labels_pred = adata.obs[latent_cell_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 3.5.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_cell_cluster_key,
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())
    
    labels = adata.obs[cell_type_key]
    labels_pred = adata.obs[latent_cell_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_cell_types': n_cell_types_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores_cell_type.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

### 3.6 shk2005-1b_1p (SlideSeqV2 Human Kidney)

In [ ]:
# Define dataset params
dataset_name = 'shk2005-1b_1p'
cell_type_key = 'cell_type'
spot_size = 30

#### 3.6.1 Load Data

In [ ]:
adata = sc.read_h5ad(f'{st_data_silver_folder_path}/{dataset_name}/Puck_200104_15_with_emb_NEMO_commongenes.h5ad')

# Filter cells without cell type labels
adata = adata[~adata.obs[cell_type_key].isna()]

In [ ]:
adata.obs[cell_type_key].value_counts()

In [ ]:
# Perform model-specific preprocessing
adata.var['feature_name'] = adata.var.index
adata.X = adata.X.toarray()

#### 3.6.2 Retrieve Embeddings

In [ ]:
# Generate zero-shot embeddings for cells
ref_embed_adata = scgpt_spatial.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    obs_to_save=adata.obs.columns.tolist(), 
    batch_size=64,
    return_new_adata=True,
)
ref_embed_adata.obsm['X'] = ref_embed_adata.X.copy()
ref_embed_adata.obsm['spatial'] = adata.obsm['spatial']
adata = ref_embed_adata

In [ ]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='X',
                key_added='X')
sc.tl.umap(adata,
           neighbors_key='X')

In [ ]:
# Visualize UMAP
sc.pl.umap(adata,
           color=cell_type_key)

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded.h5ad')

#### 3.6.3 Highlight Run (Matching Ground Truth Number of Cell Types)

In [ ]:
# Compute Leiden clustering (matching the number of clusters with the ground truth with > 100 cells -> 12)
latent_leiden_resolution = 0.9

sc.tl.leiden(adata,
             neighbors_key='X',
             key_added=latent_cell_cluster_key,
             resolution=latent_leiden_resolution)

In [ ]:
print(adata.obs[latent_cell_cluster_key].value_counts())

In [ ]:
# Save updated AnnData
adata.write_h5ad(f'{benchmarking_artifacts_folder_path}/scGPT-spatial/{dataset_name}_embedded_clustered_cell_type.h5ad')

In [ ]:
cell_type_counts = (adata.obs[cell_type_key].value_counts() < 100)
invalid_cell_types = cell_type_counts[cell_type_counts].index.tolist()

latent_cluster_counts = (adata.obs[latent_cell_cluster_key].value_counts() < 100)
invalid_clusters = latent_cluster_counts[latent_cluster_counts].index.tolist()

mask = (~adata.obs[cell_type_key].isin(invalid_cell_types)) & (~adata.obs[latent_cell_cluster_key].isin(invalid_clusters))
filtered_df = adata.obs[mask]

print(filtered_df[latent_cell_cluster_key].unique().tolist())
print(filtered_df[latent_cell_cluster_key].value_counts())

max_overlap_plot(filtered_df, cell_type_key, latent_cell_cluster_key)

In [ ]:
inferred_cell_types_ordered = [
    '0',
    '3',
    '8',
    '10',
    '2',
    '7',
    '6',
    '1',
    '4',
    '9',
    '5',
    '11',]

In [ ]:
# Plot scgpt-spatial cell types
label2celltype = {}
current_index = 0  # To keep track of the new indices

for cell_type in inferred_cell_types_ordered:
    if not isinstance(cell_type, str):
        if math.isnan(cell_type):
            continue # skip NaN values
    label2celltype[str(current_index)] = cell_type
    current_index += 1

pred_palette = {
    label2celltype.get(k, k): v
    for k, v in custom_palette.items()
}

file_name = f'{dataset_name}_scgpt-spatial_cell_types.svg'
sc.pl.spatial(adata,
              color=latent_cell_cluster_key,
              palette=pred_palette,
              spot_size=spot_size,
              save=file_name)

In [ ]:
# Compute metrics
labels = adata.obs[cell_type_key]
labels_pred = adata.obs[latent_cell_cluster_key]
labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])

nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari_score = adjusted_rand_score(labels, labels_pred)

print(f"NMI scores: {nmi_score}.")
print(f"ARI scores: {ari_score}.")

#### 3.6.4 Quantitative Performance

In [ ]:
# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

nmi_scores.append(nmi_score)
ari_scores.append(ari_score)
n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata,
                 neighbors_key='X',
                 key_added=latent_cell_cluster_key,
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata.obs[latent_cell_cluster_key].nunique())
    
    labels = adata.obs[cell_type_key]
    labels_pred = adata.obs[latent_cell_cluster_key]
    labels_pred = np.array([-1 if x is np.nan else x for x in labels_pred])
    
    nmi_score = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)

    # Store on disk
    metrics_df = pd.DataFrame({
        'leiden_resolution': latent_leiden_resolution_list[:i+2],
        'n_cell_types': n_cell_types_list,
        'NMI': nmi_scores,
        'ARI': ari_scores
    })
    metrics_df.to_csv(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_scgpt-spatial_metric_scores_cell_type.csv', index=False)

print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")

## 4. Human Niche Benchmark (Shared Protocol, 2026)

- Added on 14.10.2026 by me13. Results of the human niche-identification benchmark, computed with the shared protocol in `../human_niche_benchmark/` (see its `README.md`).
- Same cells for every method (Section 2 loading); per-section spatial graphs; Leiden on a 15-NN graph of the embedding at the **lowest resolution giving the ground-truth number of niches** (niches with > 100 cells); NMI / ARI vs. ground truth.
- Computed by `../human_niche_benchmark/import_fm.py (import_fm.lsf, METHOD=scgpt-spatial) - imports the embeddings written by Section 2 of this notebook and scores them with score.py` on LSF; this section only reads the results from `artifacts/benchmarking/human/` and recomputes nothing.
- Runs in any environment with scanpy (e.g. the `terra (sb75)` kernel).

In [ ]:
import sys
sys.path.append('../human_niche_benchmark')
from method_report import show_method_metrics, show_method_maps

show_method_metrics('scgpt-spatial')

In [ ]:
# Cluster-to-niche matching and spatial maps (ground truth vs. method) per section
show_method_maps('scgpt-spatial')